In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 84.5 MB/s eta 0:00:00


# Config

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import glob
import os
import re
from collections import deque, defaultdict
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
TEST_IMG_DIR = "/content/drive/MyDrive/CapstoneProject/make/data/batch_23"
YOLO_PATH = "/content/drive/MyDrive/CapstoneProject/models/model_26n_final/yolo/weights/best.pt"
LSTM_PATH = "/content/drive/MyDrive/CapstoneProject/models/LSTM/lstm_intent.pth"
OUTPUT_VIDEO = "/content/drive/MyDrive/CapstoneProject/test/result_lstm.mp4"
OUTPUT_VIDEO_2 = "/content/drive/MyDrive/CapstoneProject/test/result_lstm_no_zones.mp4"

# With zones

In [ ]:
STATE_WINDOW = 10
FRAME_SKIP = 3
REQ_LEN = (STATE_WINDOW - 1) * FRAME_SKIP + 1

In [ ]:
WARNING_ZONE = np.array([[200, 600], [800, 600], [700, 400], [300, 400]], np.int32)
DANGER_ZONE = np.array([[250, 640], [750, 640], [650, 500], [350, 500]], np.int32)

def natural_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', os.path.basename(filename))]

# Kiểm tra Box có chạm vào Polygon không (Dùng điểm giữa đáy của Bounding Box)
def is_in_zone(box, polygon):
    x1, y1, x2, y2 = box

    bottom_center = (int((x1 + x2) / 2), int(y2))

    # pointPolygonTest trả về > 0 nếu nằm trong, = 0 nếu nằm trên cạnh
    return cv2.pointPolygonTest(polygon, bottom_center, False) >= 0

In [ ]:
class IntentLSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolo = YOLO(YOLO_PATH)

lstm_model = IntentLSTM().to(device)
lstm_model.load_state_dict(torch.load(LSTM_PATH, map_location=device))
lstm_model.eval()

IntentLSTM(
  (lstm): LSTM(5, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

In [ ]:
image_files = sorted(glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')), key=natural_sort_key)
frame_sample = cv2.imread(image_files[0])
h_img, w_img = frame_sample.shape[:2]

# 1. WARNING ZONE (Vùng vàng - Xa hơn, rộng hơn, bao quát 2 bên làn)
w_pt1 = (int(w_img * 0.10), int(h_img * 0.80)) # Góc trái dưới
w_pt2 = (int(w_img * 0.90), int(h_img * 0.80)) # Góc phải dưới
w_pt3 = (int(w_img * 0.65), int(h_img * 0.55)) # Góc phải trên (điểm tụ xa)
w_pt4 = (int(w_img * 0.35), int(h_img * 0.55)) # Góc trái trên
WARNING_ZONE = np.array([w_pt1, w_pt2, w_pt3, w_pt4], np.int32)

# 2. DANGER ZONE (Vùng đỏ - Gần đầu xe hơn, hẹp hơn, chốt chặn va chạm)
d_pt1 = (int(w_img * 0.25), int(h_img * 0.80)) # Góc trái dưới (bằng chiều rộng xe)
d_pt2 = (int(w_img * 0.75), int(h_img * 0.80)) # Góc phải dưới (bằng chiều rộng xe)
d_pt3 = (int(w_img * 0.58), int(h_img * 0.65)) # Góc phải trên (điểm tụ gần)
d_pt4 = (int(w_img * 0.42), int(h_img * 0.65)) # Góc trái trên
DANGER_ZONE = np.array([d_pt1, d_pt2, d_pt3, d_pt4], np.int32)

out_video = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), 30, (w_img, h_img))
track_buffer = defaultdict(lambda: deque(maxlen=REQ_LEN))

In [ ]:
for i, img_path in enumerate(image_files):
    frame = cv2.imread(img_path)
    if frame is None: continue

    # --- VẼ ZONES TRÊN MÀN HÌNH ---
    # Overlay mờ cho Zone
    overlay = frame.copy()
    cv2.fillPoly(overlay, [WARNING_ZONE], (0, 255, 255)) # Vàng
    cv2.fillPoly(overlay, [DANGER_ZONE], (0, 0, 255))    # Đỏ
    cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)
    cv2.polylines(frame, [WARNING_ZONE], True, (0, 255, 255), 2)
    cv2.polylines(frame, [DANGER_ZONE], True, (0, 0, 255), 2)

    results = yolo.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        for box, tid, conf in zip(boxes, ids, confs):
            x1, y1, x2, y2 = box
            feature = [((x1+x2)/2)/w_img, ((y1+y2)/2)/h_img, (x2-x1)/w_img, (y2-y1)/h_img, float(conf)]
            track_buffer[tid].append(feature)

            # --- KIỂM TRA HẬU XỬ LÝ ZONES (RULE-BASED) ---
            in_danger = is_in_zone(box, DANGER_ZONE)
            in_warning = is_in_zone(box, WARNING_ZONE)

            intent_pred = 0 # 0: Safe, 1: Cut-in

            # --- DỰ ĐOÁN Ý ĐỊNH BẰNG LSTM ---
            if len(track_buffer[tid]) == REQ_LEN:
                sampled_feat = list(track_buffer[tid])[::FRAME_SKIP]
                state_t = torch.FloatTensor(sampled_feat).unsqueeze(0).to(device)

                with torch.no_grad():
                    output = lstm_model(state_t)
                    intent_pred = torch.argmax(output, dim=1).item()

            # --- KẾT HỢP LOGIC (HYBRID DECISION) ---
            color = (0, 255, 0) # Xanh - An toàn
            label = "Safe"

            if in_danger:
                color = (0, 0, 255)
                label = "DANGER! BRAKE!"
            elif in_warning and intent_pred == 1:
                color = (0, 165, 255) # Cam
                label = "WARNING: CUT-IN INTENT"
            elif intent_pred == 1:
                color = (0, 255, 255) # Vàng
                label = "Intent: Cut-in (Far)"
            elif in_warning:
                color = (255, 255, 0) # Xanh lơ
                label = "In Warning Zone"

            # Vẽ Box và Text
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{tid} {label}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out_video.write(frame)

out_video.release()
print(f"Hoàn tất Inference! Kết quả lưu tại {OUTPUT_VIDEO}")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 100ms
Prepared 1 package in 33ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Hoàn tất Inference! Kết quả lưu tại /content/drive/MyDrive/CapstoneProject/test/result_lstm.mp4


# No zone

In [ ]:
def natural_sort_key(filename):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', os.path.basename(filename))]

In [ ]:
class IntentLSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolo = YOLO(YOLO_PATH)

lstm_model = IntentLSTM().to(device)
lstm_model.load_state_dict(torch.load(LSTM_PATH, map_location=device))
lstm_model.eval()

IntentLSTM(
  (lstm): LSTM(5, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

In [ ]:
image_files = sorted(glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')), key=natural_sort_key)
if not image_files:
    raise ValueError("Không tìm thấy ảnh!")

frame_sample = cv2.imread(image_files[0])
h_img, w_img = frame_sample.shape[:2]

out_video = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), 30, (w_img, h_img))
track_buffer = defaultdict(lambda: deque(maxlen=REQ_LEN))

In [ ]:
for i, img_path in enumerate(image_files):
    frame = cv2.imread(img_path)
    if frame is None: continue

    results = yolo.track(frame, persist=True, verbose=False, tracker="bytetrack.yaml")

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        confs = results[0].boxes.conf.cpu().numpy()

        for box, tid, conf in zip(boxes, ids, confs):
            x1, y1, x2, y2 = box

            # Chuẩn hóa Feature
            feature = [((x1+x2)/2)/w_img, ((y1+y2)/2)/h_img, (x2-x1)/w_img, (y2-y1)/h_img, float(conf)]
            track_buffer[tid].append(feature)

            color = (0, 255, 0) # Xanh - An toàn
            label = "Safe"

            # --- DỰ ĐOÁN Ý ĐỊNH CHỈ DÙNG LSTM ---
            if len(track_buffer[tid]) == REQ_LEN:
                sampled_feat = list(track_buffer[tid])[::FRAME_SKIP]
                state_t = torch.FloatTensor(sampled_feat).unsqueeze(0).to(device)

                with torch.no_grad():
                    output = lstm_model(state_t)
                    intent_pred = torch.argmax(output, dim=1).item()

                # Nếu LSTM phát hiện có ý định cắt mặt
                if intent_pred == 1:
                    color = (0, 0, 255) # Đỏ
                    label = "CUT-IN DETECTED!"

                    # Vẽ dòng cảnh báo to giữa màn hình
                    cv2.putText(frame, "WARNING: CUT-IN!", (50, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

            # Vẽ Bounding Box và Label lên xe
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{tid} {label}", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out_video.write(frame)

out_video.release()
print(f"Hoàn tất! Kết quả lưu tại {OUTPUT_VIDEO_2}")

Hoàn tất! Kết quả lưu tại /content/drive/MyDrive/CapstoneProject/test/result_lstm_no_zones.mp4
